In [8]:
import json
import re
from pathlib import Path

INPUT_PATH = Path("../../infra/json/kg_extraction/bellicum_contract_kg.json")
OUTPUT_DIR = Path("../../infra/json/kg_extraction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "bellicum_contract_kg_normalized.json"

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '../../infra/json/kg_extraction/bellicum_contract_kg.json'

In [ ]:
def normalize_text(text):
    if not text:
        return None

    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text


In [ ]:
def normalize_id_spaces(entity_id):
    if not entity_id:
        return entity_id

    entity_id = entity_id.strip().lower()
    entity_id = re.sub(r"\s+", "_", entity_id)
    entity_id = re.sub(r"_+", "_", entity_id)

    return entity_id

In [ ]:
def normalize_party_name(name):
    if not name:
        return None

    name_norm = normalize_text(name)

    if "miltenyi" in name_norm:
        return "miltenyi"

    if "bellicum" in name_norm:
        return "bellicum"

    if "party" in name_norm:
        return "party"

    return name_norm

In [ ]:
def normalize_entity_id(ent):
    ent_type = ent["type"].lower()

    if ent["type"] == "Party":
        canonical = normalize_party_name(
            ent["properties"].get("canonical_name")
            or ent["properties"].get("name")
        )
        return f"party_{canonical}"

    if ent["type"] == "DefinedTerm":
        term = normalize_text(ent["properties"].get("term"))
        return f"definedterm_{term}"

    if ent["type"] == "Value":
        val = normalize_text(ent["label"])
        return f"value_{val}"

    if ent["type"] == "Clause":
        return ent["properties"]["clause_id"]

    # default
    return ent["id"]


In [ ]:
WEAK_ACTIONS = [
    "desire",
    "desires",
    "intend",
    "intends",
    "wish",
    "wishes",
    "expect",
    "expects",
]

def is_weak_event(ent):
    if ent["type"] not in ["Obligation", "Right", "Permission"]:
        return False

    action = ent["properties"].get("normalized_action")

    if not action:
        return False

    return action in WEAK_ACTIONS


In [ ]:
entities = data["knowledge_graph"]["entities"]
relations = data["knowledge_graph"]["relations"]

new_entities = {}
id_map = {}

# ---- normalize entities ----

for ent in entities:
    if is_weak_event(ent):
        continue

    new_id = normalize_id_spaces(normalize_entity_id(ent))
    old_id = ent["id"]

    id_map[old_id] = new_id

    # normalize properties
    props = ent.get("properties", {})

    if ent["type"] == "Party":
        props["canonical_name"] = normalize_party_name(
            props.get("canonical_name") or props.get("name")
        )

    if "normalized_action" in props:
        props["normalized_action"] = normalize_text(props["normalized_action"])

    if "normalized_object" in props:
        props["normalized_object"] = normalize_text(props["normalized_object"])

    # merge duplicates
    if new_id not in new_entities:
        ent["id"] = new_id
        ent["properties"] = props
        new_entities[new_id] = ent
    else:
        # merge evidence
        old_ev = new_entities[new_id].get("evidence_text", [])
        new_ev = ent.get("evidence_text", [])

        if not isinstance(old_ev, list):
            old_ev = [old_ev]

        if not isinstance(new_ev, list):
            new_ev = [new_ev]

        merged = list(set(old_ev + new_ev))
        new_entities[new_id]["evidence_text"] = merged


# ---- normalize relations ----

new_relations = []

for rel in relations:
    src = rel["source"]
    tgt = rel["target"]

    if src not in id_map or tgt not in id_map:
        continue

    rel = dict(rel)
    rel["source"] = normalize_id_spaces(id_map[src])
    rel["target"] = normalize_id_spaces(id_map[tgt])

    new_relations.append(rel)


# ----------------------------------
# OUTPUT
# ----------------------------------

normalized_kg = {
    "mode": "knowledge_graph",
    "metadata": data["metadata"],
    "knowledge_graph": {
        "entities": list(new_entities.values()),
        "relations": new_relations,
    }
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(normalized_kg, f, indent=2, ensure_ascii=False)

print("Saved normalized KG:", OUTPUT_PATH)
print("Entities:", len(new_entities))
print("Relations:", len(new_relations))

Saved normalized KG: ../../infra/json/kg_extraction/bellicum_contract_kg_normalized.json
Entities: 19
Relations: 29
